# Reto de Laboratorio: Preparando la Matriz Tensorial de la Liga ⚽🤖


Contexto de Negocio:
El equipo de analítica quedó fascinado con el Almacén de Datos (OLAP) que construyeron la semana pasada. Ahora, el departamento de Inteligencia Artificial quiere dar el siguiente paso: entrenar una Red Neuronal para predecir cuántos goles anotará un jugador la próxima temporada.El Científico de Datos les acaba de enviar la tabla resultante del cubo OLAP, pero se detuvo por un error de Scikit-Learn: el algoritmo no sabe leer posiciones como "Forward" ni jerarquías como "Oro".Su misión como Ingenieros de Machine Learning es aplicar Feature Engineering para traducir esta tabla humana a una Matriz $\mathbf{X}$ puramente matemática, sin destruir la lógica vectorial.

In [12]:
import pandas as pd
import numpy as np

# 1. Ingesta de datos desde el repositorio crudo
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/players_data-2025_2026.csv"
df_scouting = pd.read_csv(url)

# 2. SIMULACIÓN: El equipo de analistas agregó una columna de jerarquía de talento
# (Evaluación cualitativa del jugador)
np.random.seed(42)
df_scouting['scout_tier'] = np.random.choice(['Bronce', 'Plata', 'Oro', 'Diamante'], size=len(df_scouting))
#
# # Filtramos las columnas que usaremos para el modelo predictivo
columnas_ml = ['Player', 'Squad', 'Age', 'Pos', 'scout_tier', 'MP', 'Gls']
matriz_base = df_scouting[columnas_ml].copy()
#
print("--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---")
display(matriz_base.head())

--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---


,Player,Squad,Age,Pos,scout_tier,MP,Gls
0,Brenden Aaronson,Leeds United,24.0,"MF,FW",Oro,37,4
1,Jerome Abbey,Wolves,15.0,MF,Diamante,1,0
2,Zach Abbott,Nottingham Forest,19.0,DF,Bronce,3,0
3,Jones El-Abdellaoui,Celta Vigo,19.0,"MF,FW",Oro,22,2
4,Himad Abdelli,Marseille,25.0,MF,Oro,8,0


### FASE 1: El Traductor Jerárquico (Label Encoding)

Observa la columna `scout_tier`. Contiene valores de texto: 'Bronce', 'Plata', 'Oro', 'Diamante'. Esta variable tiene un **orden natural** claro (Diamante es superior a Bronce).

**Tu tarea:**
Crea un diccionario de mapeo que asigne números enteros respetando la jerarquía natural del talento (iniciando en 1). Luego, aplica este mapeo a la base de datos para crear una nueva columna llamada `scout_encoded`.

In [14]:
scout_mapping = {
    'Bronce' : 1,
    'Plata' : 2,
    'Oro' : 3,
    'Diamante' : 4,
}

matriz_base['scout_encoded'] =matriz_base['scout_tier'].map(scout_mapping)

print("Resultado del Label Encoding:")
display(matriz_base[['Player', 'scout_tier', 'scout_encoded']].head())

Resultado del Label Encoding:


,Player,scout_tier,scout_encoded
0,Brenden Aaronson,Oro,3
1,Jerome Abbey,Diamante,4
2,Zach Abbott,Bronce,1
3,Jones El-Abdellaoui,Oro,3
4,Himad Abdelli,Oro,3


### FASE 2: El Traductor Nominal (One-Hot Encoding)

Ahora observa la columna `position`. Contiene valores como 'Forward' (Delantero), 'Midfielder' (Mediocampista), 'Defender' (Defensa), etc.
Esta variable es **Nominal**. Si le asignamos números (1 al Delantero y 3 al Defensa), la Red Neuronal creerá falsamente que un Defensa "vale el triple" que un Delantero.

**Tu tarea:**
Utiliza la función de Pandas correspondiente para destruir la columna `position` y expandirla en múltiples columnas booleanas (0 y 1). Guarda el resultado en una nueva variable llamada `matriz_X`.

In [23]:
# CÓDIGO ESQUELETO PARA EL ALUMNO:
posiciones_dummies = matriz_base['Pos'].str.get_dummies(sep=',')

matriz_X = pd.concat([matriz_base.drop(columns=['Pos']), posiciones_dummies], axis=1)

print("Resultado del One-Hot Encoding:")
display(matriz_X.head())

Resultado del One-Hot Encoding:


,Player,Squad,Age,scout_tier,MP,Gls,scout_encoded,DF,FW,GK,MF
0,Brenden Aaronson,Leeds United,24.0,Oro,37,4,3,0,1,0,1
1,Jerome Abbey,Wolves,15.0,Diamante,1,0,4,0,0,0,1
2,Zach Abbott,Nottingham Forest,19.0,Bronce,3,0,1,1,0,0,0
3,Jones El-Abdellaoui,Celta Vigo,19.0,Oro,22,2,3,0,1,0,1
4,Himad Abdelli,Marseille,25.0,Oro,8,0,3,0,0,0,1


### FASE 3: Purgando la Basura Humana

La Inteligencia Artificial está casi lista para devorar los datos. Sin embargo, si le enviamos la matriz ahora mismo, el algoritmo fallará porque todavía hay texto puro estorbando.

**Tu tarea:**
Elimina de `matriz_X` todas las columnas que ya no le sirven al algoritmo:
1. Las columnas de texto original que ya transformaste (como `scout_tier`).
2. Las columnas descriptivas/operativas que no aportan valor matemático universal (`player_name`, `team_name`).

In [24]:
columnas_ruido = ['Player', 'Squad', 'scout_tier']
matriz_X = matriz_X.drop(columns=columnas_ruido)

print("--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---")
display(matriz_X.head())
print("\nTipos de datos finales (Todos deben ser numéricos):")
print(matriz_X.dtypes)

--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---


,Age,MP,Gls,scout_encoded,DF,FW,GK,MF
0,24.0,37,4,3,0,1,0,1
1,15.0,1,0,4,0,0,0,1
2,19.0,3,0,1,1,0,0,0
3,19.0,22,2,3,0,1,0,1
4,25.0,8,0,3,0,0,0,1



Tipos de datos finales (Todos deben ser numéricos):
Age              float64
MP                 int64
Gls                int64
scout_encoded      int64
DF                 int64
FW                 int64
GK                 int64
MF                 int64
dtype: object


### 🧠 FASE 4: Defensa Arquitectónica (Responde aquí mismo)

Imagina que el CTO de la empresa revisa tu código y te hace dos preguntas críticas. Responde con argumentos de ingeniería:

**Pregunta 1:** *"Noté que borraste la columna `team_name` en lugar de aplicarle One-Hot Encoding. En nuestra base de datos mundial de la FIFA tenemos 4,000 equipos diferentes. Si le hubieras aplicado One-Hot Encoding a la columna de equipos, ¿qué le habría pasado a la estructura matemática de tu Matriz X y a la memoria RAM de nuestro servidor?"*
> **Tu Respuesta:** La estructura matematica hubiera crecido demasiado hacia niveles donde la RAM no seria suficiente en una peticion, lo que nos hace una matriz donde la gran mayoria de esos 4000 valores serian 0.

**Pregunta 2:** *"¿Por qué usamos Label Encoding (1, 2, 3, 4) para la evaluación del scout (`scout_tier`), pero no usamos esa misma técnica matemática para la posición en el campo (`position`)?"*
> **Tu Respuesta:** Ya que el scout tier es una variable la cual es cualitativa, podemos asumir que un scout nivel bronce es peor que uno diamante, y se pueden ver como niveles de jerarquia. Mientras que position, aunque tambien es cualitativa no existe una jerarquia como tal dentro de lo que es el mundo del futbol, donde una posicion tenga mejor valor que otra, ya que ambas posiciones son indispensables para el juego. Ademas que de crearlo de esa manera podria llegar a sesgar a nuestro modelo haciendolo pensar que tal vez un gran portero no va a ser igual un delantero que tenga por lo menos 1 gol, cuando por posiciones el portero usualmente no tendria goles.